In [1]:
import pandas as pd
import miceforest as mf
import matplotlib.pyplot as plt
import numpy as np
import os
from utils.functions import interp_and_wavg


%matplotlib qt

In [2]:
os.chdir(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805')

In [3]:
df_lab = pd.read_parquet('lab_20240805.parquet')
df_lab.studyId_0831 = df_lab.studyId_0831.astype('uint64')

In [ ]:
df_lab.groupby('studyId_0831').effectiveDateTime.nunique()

In [5]:
df_consult = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240822\consult_20240822.sas7bdat',
        encoding='latin1')
df_consult.studyId_0831 = df_consult.studyId_0831.astype('uint64')

In [6]:
df_consult = df_consult.drop(['Consult_id', 'ConsultStatus', 'type_code_original',
                              'type_display_original','created', 'title', 'author_usercode',
                              'initial_author_usercode', 'custodian_Organization_value',
                              'SpecialismeNaam', 'initial_custodian_Organization_v',
                              'SpecialismeNaamInitial', 'LocationCode', 'LocationName',
                              'PatientProtected', 'section_title_code_original',
                              'section_title_display_original', 'section_text',
                              'section_text_contentType', 'orderedBy', 'index_date'],axis='columns')

In [ ]:
df_consult = df_consult.groupby('studyId_0831').last()
df_consult

Let's remove uneuseful columns

In [8]:
df_lab = df_lab.drop(['identifier_value', 'basedOn_ProcedureRequest_value',
                      'LabID', 'category3_display_original', 'category4_code_original',
                      'code_code_original', 'code2_code_original',
                      'code2_display_original', 'code3_display_original',
                      'interpretation_display','referenceRange_text','valueQuantity_code_original',
                      'code3_system_original', 'code4_display_original', 'comment',
                      'status_display_original',
                      'valueQuantity_comparator', 'valueQuantity_unit',
                      'valueQuantity_unit_original', 'valueString',
                      'Practitioner_Rol', 'gender', 'organization_Organization_value',
                      'specialty_specialtyAGB_display', 'index_date'], axis='columns')

Storing the reference range of values per substance

In [ ]:
substances = pd.read_excel(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\excel\measurements_ranges.xlsx')
substances = substances.set_index('substance')


substances.loc['Kreatinine']['unit']

Pivoting on the chemical sub

In [ ]:
df_lab = df_lab.sort_values(by='studyId_0831').reset_index(drop=True)
df_lab

In [ ]:
df_merged = pd.merge(df_lab, df_consult, right_index=True,left_on='studyId_0831',how='left')
df_merged

In [ ]:
df_merged['delta'] = (df_merged.date - df_merged.effectiveDateTime).dt.days
df_merged

In [13]:
df_merged = df_merged.query('delta < 365 and delta > -10')

---

In [ ]:
df_lab_p = df_merged.pivot(
    columns='code_display_original', values='valueQuantity_value')
df_lab_p

In [ ]:
df_lab_p['studyId_0831'] = df_merged.studyId_0831
df_lab_p['studyId_0831'] = df_lab_p['studyId_0831'].astype('uint64')
df_lab_p

In [ ]:
df_lab_p[df_lab_p.columns[:-1]].plot(kind='box',subplots=True)

In [17]:
for c in df_lab_p[df_lab_p.columns[:-1]].columns:
    mu = df_lab_p[c].mean()
    si = df_lab_p[c].std()
    df_lab_p[c] = df_lab_p[c].apply(lambda x: np.nan if x > mu + (3 * si) or x < mu - (3 * si) else x)

In [ ]:
df_lab_p[df_lab_p.columns[:-1]].plot(kind='box',subplots=True)

In [19]:
def weighted_avg(x : pd.Series):
    if x.count() == 0:
        return np.nan
    else:
        w = 1/np.abs(df_merged.loc[x[x.notnull()].index].delta)
        w = w/w.sum()
        return np.average(x[x.notnull()], weights=w)

In [ ]:
df_lab_p_g = df_lab_p.groupby('studyId_0831', observed=True).agg(weighted_avg)
df_lab_p_g

In [ ]:
df_lab_p_g.notna().sum()

In [22]:
ker = mf.ImputationKernel(
    df_lab_p_g.reset_index(),
    num_datasets=5,
    save_all_iterations_data=True,
    random_state=42)

ker.mice(iterations=10,
         n_estimators=200,
         boosting='gbdt',
         min_sum_hessian_in_leaf=60)

In [ ]:
ker.plot_imputed_distributions()

In [24]:
df_p_complete = ker.complete_data()

In [25]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer,KNNImputer
from sklearn.ensemble import RandomForestRegressor
import seaborn as sns

# Create the imputer object
imputer = KNNImputer()
# Fit and transform the data
data_imputed = imputer.fit_transform(df_lab_p_g)
# Convert back to DataFrame
data_imputed = pd.DataFrame(data_imputed, columns=df_lab_p_g.columns)


In [37]:
# Set the number of rows and columns for subplots
num_features = data_imputed.select_dtypes(include=[np.number]).columns
num_plots = len(num_features)
rows = (num_plots // 3) + (num_plots % 3 > 0)

# Set up the figure
plt.figure(figsize=(20,int(1.5*rows)))

for i, feature_name in enumerate(num_features):
    plt.subplot(rows, 3, i + 1)
    sns.histplot(df_lab_p_g.reset_index()[feature_name], discrete=False, kde=True, color='#000000', label='Original Data', stat='density',alpha=0)
    sns.histplot(data_imputed[feature_name], discrete=False, kde=True, color='DarkRed', label='Imputed Data', stat='density', alpha=0)
    plt.title(f'Distribution of {feature_name}')
    plt.xlabel(feature_name)
    plt.ylabel('Density')
    #plt.legend()

plt.tight_layout()
plt.show()